# Qwen3.5-4B Fashion Agent — 864건 5 LOOK LoRA 재학습 및 100건 확장 평가 (v2)

기존 `v0.1` 학습 데이터(864건)는 그대로 보존하면서, **과적합 억제 및 표현 다양성 향상 기법(1 epoch)**과 **확장 평가(Validation 100건 / Test 100건 + 기존 8건 회귀 비교)**를 통합한 실험 노트북입니다.

### 주요 개선 사항
1. **1 Epoch 과적합 방지 & 표현 품질 개선**:
   - **Completion-Only Loss (`train_on_responses_only`)**: 긴 시스템/컨텍스트 입력이 아닌 `assistant` 출력 구간에만 Loss를 집중 계산
   - **NEFTune (`neftune_noise_alpha=5`)**: 임베딩 노이즈 주입으로 문장 암기 억제 및 코디 사유(`reason`) 다양성 확보
   - **균등 셔플링 (`seed=42`)**: 패션 추천(540건)과 Tool Calling(324건)을 전체 step에 균일 배치
   - **Cosine 스케줄러 & Weight Decay (`0.01`)**: 안정적 수렴 및 가중치 편향 완화
2. **실제 토큰 길이 & 중복 무결성 사전 검증**: 중첩 리스트 길이 버그 수정(`4,096` 토큰 초과 여부 정확 검사) 및 `train` ↔ `validation`/`test` 중복 제로 검증
3. **속도 사전 측정 & 중단 후 재개(Resume) 지원**:
   - 전체 평가 전 검증 4건으로 생성 속도와 예상 소요 시간 측정
   - 사례별 결과를 `.jsonl`로 즉시 저장하고, 이미 완료된 케이스는 건너뛰어 Colab 세션 끊김에도 안전하게 재개


In [ ]:
# Colab 런타임을 T4 GPU로 설정한 뒤 실행합니다.
!pip install -qU unsloth datasets trl accelerate bitsandbytes


## 1. 데이터 업로드 (`qwen35_864_five_look_data_v2.zip`)


In [ ]:
from google.colab import files
from pathlib import Path
import zipfile

uploaded = files.upload()
zip_name = next(
    (name for name in uploaded if name.endswith(".zip")),
    None,
)
assert zip_name is not None, "qwen35_864_five_look_data_v2.zip 파일을 업로드하세요."

data_root = Path("/content/qwen35_864_five_look_data_v2")
data_root.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_name) as archive:
    archive.extractall(data_root)

required_files = [
    "train.jsonl",
    "validation.jsonl",
    "test.jsonl",
    "validation_legacy_8.jsonl",
    "test_legacy_8.jsonl",
    "dataset_manifest.json",
]
for required in required_files:
    assert (data_root / required).exists(), f"누락된 파일: {required}"

print("데이터 준비 완료:")
for path in sorted(data_root.iterdir()):
    print("-", path.name)


## 2. 실행 설정, 무결성(SHA256) 및 중복 제로 검증


In [ ]:
import hashlib
import json
import re
import time
from datetime import datetime, timezone
from pathlib import Path

import torch
from datasets import load_dataset

# === 실험 및 실행 제어 설정 ===
SEED = 42
MODEL_NAME = "unsloth/Qwen3.5-4B"
MAX_SEQ_LENGTH = 4096
MAX_NEW_TOKENS = 1600
NUM_EPOCHS = 1

# 실행 모드 스위치 (시간 관리용)
RUN_SPEED_BENCHMARK = True       # 검증 4건으로 추론 속도 사전 측정
BENCHMARK_COUNT = 4
RUN_LEGACY_8_EVAL = True         # 기존 Test 8건 회귀 비교 실행
RUN_NEW_100_EVAL = True          # 신규 Test 100건 전체 실행 (시간 부족 시 False로 두고 필요할 때 켜기)
MAX_NEW_100_LIMIT = None         # 일부만 먼저 돌릴 때 정수 지정 (예: 20), 전체는 None

OUTPUT_DIR = "/content/qwen35-fashion-864-v2"
ADAPTER_DIR = "/content/qwen35-fashion-864-lora-v2"
CACHE_DIR = Path("/content/eval_cache_v2")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

manifest = json.loads(
    (data_root / "dataset_manifest.json").read_text(encoding="utf-8")
)

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

for file_name, expected_hash in manifest["sha256"].items():
    actual_hash = sha256(data_root / file_name)
    assert actual_hash == expected_hash, f"{file_name} SHA256 불일치: {actual_hash}"

dataset = load_dataset(
    "json",
    data_files={
        "train": str(data_root / "train.jsonl"),
        "validation": str(data_root / "validation.jsonl"),
        "test": str(data_root / "test.jsonl"),
        "validation_legacy": str(data_root / "validation_legacy_8.jsonl"),
        "test_legacy": str(data_root / "test_legacy_8.jsonl"),
    },
)

expected_counts = {
    "train": 864,
    "validation": 100,
    "test": 100,
    "validation_legacy": 8,
    "test_legacy": 8,
}
for split_name, expected_count in expected_counts.items():
    assert len(dataset[split_name]) == expected_count, (
        f"{split_name} 개수 오류: {len(dataset[split_name])}"
    )

# 중복(데이터 누수) 검사
def get_user_prompts(ds_split):
    prompts = []
    for row in ds_split:
        u = next(m["content"] for m in row["messages"] if m["role"] == "user")
        prompts.append(u)
    return set(prompts)

train_prompts = get_user_prompts(dataset["train"])
val_prompts = get_user_prompts(dataset["validation"])
test_prompts = get_user_prompts(dataset["test"])

assert len(train_prompts & val_prompts) == 0, "Train-Validation 중복 발견!"
assert len(train_prompts & test_prompts) == 0, "Train-Test 중복 발견!"
assert len(val_prompts & test_prompts) == 0, "Validation-Test 중복 발견!"

# FR / TC 균등 셔플링
dataset["train"] = dataset["train"].shuffle(seed=SEED)

print("SHA256 무결성 및 중복 제로 검증 통과!")
print(dataset)


## 3. Qwen3.5-4B 모델 로드 (FP16)


In [ ]:
from unsloth import FastLanguageModel

model, processor = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=False,
    load_in_16bit=True,
    full_finetuning=False,
)

print("model dtype:", next(model.parameters()).dtype)
print("CUDA 사용 가능:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("allocated GB:", round(torch.cuda.memory_allocated() / 1024**3, 2))


## 4. Chat Template 적용 및 실제 토큰 길이 정밀 검사


In [ ]:
def render_record(example, add_generation_prompt=False, prompt_only=False):
    messages = example["messages"]

    if prompt_only:
        first_assistant = next(
            (
                index
                for index, message in enumerate(messages)
                if message["role"] == "assistant"
            ),
            len(messages),
        )
        messages = messages[:first_assistant]

    kwargs = {
        "tokenize": False,
        "add_generation_prompt": add_generation_prompt,
        "enable_thinking": False,
    }

    tools = example.get("tools") or []
    if tools:
        kwargs["tools"] = tools

    return processor.apply_chat_template(messages, **kwargs)


def add_text(example):
    return {"text": render_record(example)}


formatted = dataset.map(add_text)


def actual_token_length(text):
    encoded = processor(text=text, add_special_tokens=False)["input_ids"]
    if encoded and isinstance(encoded[0], list):
        return len(encoded[0])
    return len(encoded)


length_report = {}
for split_name in ["train", "validation", "test", "validation_legacy", "test_legacy"]:
    lengths = [actual_token_length(text) for text in formatted[split_name]["text"]]
    over_limit = sum(1 for l in lengths if l > MAX_SEQ_LENGTH)
    length_report[split_name] = {
        "count": len(lengths),
        "min": min(lengths),
        "max": max(lengths),
        "mean": round(sum(lengths) / len(lengths), 1),
        "over_4096": over_limit,
    }
    print(f"[{split_name}]", length_report[split_name])
    assert over_limit == 0, f"{split_name}에 {MAX_SEQ_LENGTH} 토큰 초과 데이터가 {over_limit}건 있습니다."

print("\n샘플 렌더링 미리보기 (앞 800자):")
print(formatted["train"][0]["text"][:800])


## 5. 추론 엔진(캐싱·재개 지원) 및 속도 사전 측정


In [ ]:
EXPECTED_ROLES = [
    "preference_best",
    "preference_alternative",
    "exploration",
    "wardrobe_best",
    "wardrobe_alternative",
]


def record_kind(row):
    system_text = row["messages"][0].get("content", "")
    if "패션 코디 생성기" in system_text:
        return "fashion_recommendation"
    return "tool_calling"


def expected_first_action(row):
    kind = record_kind(row)
    if kind == "fashion_recommendation":
        return "fashion_recommendation"

    first_assistant = next(
        message
        for message in row["messages"]
        if message["role"] == "assistant"
    )
    tool_calls = first_assistant.get("tool_calls") or []
    if not tool_calls:
        return "direct_answer"
    return tool_calls[0]["function"]["name"]


def run_eval_with_resume(
    current_model,
    rows,
    prefix,
    cache_jsonl_path,
    max_new_tokens=MAX_NEW_TOKENS,
):
    FastLanguageModel.for_inference(current_model)
    cache_path = Path(cache_jsonl_path)
    cached_map = {}

    if cache_path.exists():
        for line in cache_path.read_text(encoding="utf-8").splitlines():
            if line.strip():
                item = json.loads(line)
                cached_map[item["id"]] = item
        print(f"기존 캐시 로드 ({cache_path.name}): {len(cached_map)}건 완료됨")

    results = []
    newly_run = 0
    t0 = time.time()

    with cache_path.open("a", encoding="utf-8") as writer:
        for index, row in enumerate(rows, start=1):
            case_id = f"{prefix}-{index:03d}"
            if case_id in cached_map:
                results.append(cached_map[case_id])
                continue

            kind = record_kind(row)
            prompt = render_record(
                row,
                add_generation_prompt=True,
                prompt_only=True,
            )
            inputs = processor(
                text=prompt,
                return_tensors="pt",
                add_special_tokens=False,
            ).to("cuda")

            # 태스크별 추론 파라미터 분리:
            # - Tool Calling: 결정론적(do_sample=False)으로 JSON/인자 정확도 극대화
            # - 5 LOOK 생성: 약간의 온도(0.35)와 반복 페널티(1.08)로 문장 다양성 확보
            gen_kwargs = dict(
                max_new_tokens=max_new_tokens,
                use_cache=True,
            )
            if kind == "fashion_recommendation":
                gen_kwargs.update(
                    do_sample=True,
                    temperature=0.35,
                    top_p=0.9,
                    repetition_penalty=1.08,
                )
            else:
                gen_kwargs.update(do_sample=False)

            step_start = time.time()
            with torch.inference_mode():
                output = current_model.generate(**inputs, **gen_kwargs)

            generated = output[:, inputs["input_ids"].shape[1]:]
            response = processor.batch_decode(
                generated,
                skip_special_tokens=True,
            )[0]
            elapsed = round(time.time() - step_start, 2)

            record_out = {
                "id": case_id,
                "type": kind,
                "expected_first_action": expected_first_action(row),
                "elapsed_sec": elapsed,
                "response": response,
            }
            writer.write(json.dumps(record_out, ensure_ascii=False) + "\n")
            writer.flush()
            results.append(record_out)
            newly_run += 1

            if newly_run % 5 == 0 or index == len(rows):
                avg_sec = (time.time() - t0) / newly_run
                rem = (len(rows) - index) * avg_sec
                print(
                    f"[{case_id}] 완료 ({elapsed}s) | 신규 {newly_run}건 평균 {avg_sec:.1f}s/건 | 잔여 예상 {rem/60:.1f}분"
                )

    return results


if RUN_SPEED_BENCHMARK:
    # FR 2건 + TC 2건 샘플로 속도 측정
    val_fr_sample = [r for r in dataset["validation"] if record_kind(r) == "fashion_recommendation"][:2]
    val_tc_sample = [r for r in dataset["validation"] if record_kind(r) == "tool_calling"][:2]
    bench_rows = val_fr_sample + val_tc_sample

    bench_start = time.time()
    bench_res = run_eval_with_resume(
        model,
        bench_rows,
        prefix="BENCH",
        cache_jsonl_path=CACHE_DIR / "speed_benchmark.jsonl",
    )
    bench_total = time.time() - bench_start
    avg_per_item = bench_total / len(bench_rows)
    print(f"\n[속도 측정 결과] 4건 총 {bench_total:.1f}초 (평균 {avg_per_item:.1f}초/건)")
    print(f"-> Test 100건 단방향 예상 소요 시간: 약 {(avg_per_item * 100) / 60:.1f}분")


## 6. Base Model 평가 (기존 8건 + 신규 100건 캐싱 실행)


In [ ]:
legacy_test_rows = [dataset["test_legacy"][i] for i in range(len(dataset["test_legacy"]))]
new_test_rows = [dataset["test"][i] for i in range(len(dataset["test"]))]
if MAX_NEW_100_LIMIT is not None:
    new_test_rows = new_test_rows[:MAX_NEW_100_LIMIT]

baseline_legacy_results = []
if RUN_LEGACY_8_EVAL:
    baseline_legacy_results = run_eval_with_resume(
        model,
        legacy_test_rows,
        prefix="LEGACY",
        cache_jsonl_path=CACHE_DIR / "baseline_legacy_8.jsonl",
    )

baseline_new_results = []
if RUN_NEW_100_EVAL:
    baseline_new_results = run_eval_with_resume(
        model,
        new_test_rows,
        prefix="TEST100",
        cache_jsonl_path=CACHE_DIR / "baseline_new_100.jsonl",
    )

Path("/content/baseline_results.json").write_text(
    json.dumps(
        {"legacy_8": baseline_legacy_results, "new_100": baseline_new_results},
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)
print("Base 결과 저장 완료: /content/baseline_results.json")


## 7. LoRA 설정 및 과적합 억제 Trainer 구성 (1 Epoch)


In [ ]:
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
    max_seq_length=MAX_SEQ_LENGTH,
)

model.print_trainable_parameters()

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=2e-4,
    warmup_ratio=0.05,
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    neftune_noise_alpha=5,
    optim="adamw_8bit",
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    packing=False,
    dataset_num_proc=1,
    report_to="none",
    seed=SEED,
)

trainer_kwargs = dict(
    model=model,
    train_dataset=formatted["train"],
    eval_dataset=formatted["validation"],
    args=training_args,
)

try:
    trainer = SFTTrainer(
        processing_class=processor,
        **trainer_kwargs,
    )
except TypeError:
    trainer = SFTTrainer(
        tokenizer=processor,
        **trainer_kwargs,
    )

# Assistant 응답 구간만 Loss 계산 (입력 컨텍스트 암기 방지)
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

# 마스킹 정상 작동 검증 (전체가 -100으로 마스킹되지 않았는지 확인)
sample_labels = trainer.train_dataset[0]["labels"]
active_tokens = sum(1 for lbl in sample_labels if lbl != -100)
total_tokens = len(sample_labels)
assert active_tokens > 0, "train_on_responses_only 마스킹 오류: 활성 토큰이 0개입니다."
print(f"Completion-Only Loss 마스킹 확인: 전체 {total_tokens} 토큰 중 Assistant 응답 {active_tokens} 토큰만 학습")


## 8. 1 Epoch 학습 실행


In [ ]:
training_started_at = datetime.now(timezone.utc)
train_result = trainer.train()
training_finished_at = datetime.now(timezone.utc)

print(train_result.metrics)

model.save_pretrained(ADAPTER_DIR)
processor.save_pretrained(ADAPTER_DIR)

training_report = {
    "started_at_utc": training_started_at.isoformat(),
    "finished_at_utc": training_finished_at.isoformat(),
    "duration_seconds": (training_finished_at - training_started_at).total_seconds(),
    "model_name": MODEL_NAME,
    "epochs": NUM_EPOCHS,
    "lr_scheduler_type": "cosine",
    "weight_decay": 0.01,
    "neftune_noise_alpha": 5,
    "completion_only_loss": True,
    "max_seq_length": MAX_SEQ_LENGTH,
    "train_count": len(dataset["train"]),
    "validation_count": len(dataset["validation"]),
    "train_metrics": train_result.metrics,
    "log_history": trainer.state.log_history,
    "length_report": length_report,
}

Path("/content/training_metrics.json").write_text(
    json.dumps(training_report, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("LoRA v2 저장 완료:", ADAPTER_DIR)
print("학습 로그 저장 완료: /content/training_metrics.json")


## 9. Fine-tuned Model 평가 (기존 8건 + 신규 100건 캐싱 실행)


In [ ]:
finetuned_legacy_results = []
if RUN_LEGACY_8_EVAL:
    finetuned_legacy_results = run_eval_with_resume(
        model,
        legacy_test_rows,
        prefix="LEGACY",
        cache_jsonl_path=CACHE_DIR / "finetuned_legacy_8.jsonl",
    )

finetuned_new_results = []
if RUN_NEW_100_EVAL:
    finetuned_new_results = run_eval_with_resume(
        model,
        new_test_rows,
        prefix="TEST100",
        cache_jsonl_path=CACHE_DIR / "finetuned_new_100.jsonl",
    )

Path("/content/finetuned_results.json").write_text(
    json.dumps(
        {"legacy_8": finetuned_legacy_results, "new_100": finetuned_new_results},
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)
print("Fine-tuned 결과 저장 완료: /content/finetuned_results.json")


## 10. 자동 정밀 채점 (슬롯 중복·옷장 ID 검증 포함) 및 분리 리포트


In [ ]:
def extract_json(text):
    start = text.find("{")
    end = text.rfind("}")
    if start < 0 or end < start:
        return None
    try:
        return json.loads(text[start:end + 1])
    except json.JSONDecodeError:
        return None


def extract_first_tool(text):
    patterns = [
        r"<function=([a-zA-Z0-9_]+)>",
        r'"name"\s*:\s*"([a-zA-Z0-9_]+)"',
    ]
    for pattern in patterns:
        match = re.search(pattern, text)
        if match:
            return match.group(1)
    return None


def extract_wardrobe_ids_from_row(row):
    user_txt = next(m["content"] for m in row["messages"] if m["role"] == "user")
    if "context: " not in user_txt:
        return set()
    ctx = json.loads(user_txt.split("context: ", 1)[1])
    return {item["item_id"] for item in ctx.get("wardrobe", [])}


def score_result(item, raw_row):
    response = item["response"]
    expected = item["expected_first_action"]

    if item["type"] == "fashion_recommendation":
        parsed = extract_json(response)
        looks = parsed.get("looks") if isinstance(parsed, dict) else None
        roles = [look.get("role") for look in looks] if isinstance(looks, list) else []
        look_ids = [look.get("look_id") for look in looks] if isinstance(looks, list) else []

        # 추가 계약 검증: 슬롯 중복 없음 & 옷장 ID 정확성
        no_dup_slots = False
        wardrobe_id_valid = False
        if isinstance(looks, list) and len(looks) == 5:
            valid_wids = extract_wardrobe_ids_from_row(raw_row)
            slot_checks = []
            wid_checks = []
            for look in looks:
                if look.get("status") == "unavailable":
                    slot_checks.append(True)
                    wid_checks.append(look.get("items") == [])
                else:
                    items = look.get("items") or []
                    slots = [it.get("slot") for it in items]
                    slot_checks.append(len(slots) == len(set(slots)) and len(items) > 0)
                    if look.get("look_id") in (1, 2, 3):
                        wid_checks.append(all(it.get("item_id") is None for it in items))
                    else:
                        wid_checks.append(all(it.get("item_id") in valid_wids for it in items))
            no_dup_slots = all(slot_checks)
            wardrobe_id_valid = all(wid_checks)

        return {
            "json_valid": parsed is not None,
            "five_looks": isinstance(looks, list) and len(looks) == 5,
            "roles_correct": roles == EXPECTED_ROLES,
            "look_ids_correct": look_ids == [1, 2, 3, 4, 5],
            "no_duplicate_slots": no_dup_slots,
            "wardrobe_ids_valid": wardrobe_id_valid,
            "first_action_correct": None,
        }

    parsed = extract_json(response)
    if expected == "direct_answer":
        return {
            "json_valid": parsed is not None,
            "five_looks": None,
            "roles_correct": None,
            "look_ids_correct": None,
            "no_duplicate_slots": None,
            "wardrobe_ids_valid": None,
            "first_action_correct": (
                extract_first_tool(response) is None
                and isinstance(parsed, dict)
                and parsed.get("type") == "direct_answer"
            ),
        }

    return {
        "json_valid": None,
        "five_looks": None,
        "roles_correct": None,
        "look_ids_correct": None,
        "no_duplicate_slots": None,
        "wardrobe_ids_valid": None,
        "first_action_correct": extract_first_tool(response) == expected,
    }


def build_split_comparison(base_list, fine_list, raw_rows):
    comp = []
    for before, after, raw_row in zip(base_list, fine_list, raw_rows):
        comp.append(
            {
                "id": before["id"],
                "type": before["type"],
                "expected_first_action": before["expected_first_action"],
                "baseline_score": score_result(before, raw_row),
                "finetuned_score": score_result(after, raw_row),
                "baseline_response": before["response"],
                "finetuned_response": after["response"],
            }
        )
    return comp


def summarize_scores(comp_list):
    metrics = [
        "json_valid",
        "five_looks",
        "roles_correct",
        "look_ids_correct",
        "no_duplicate_slots",
        "wardrobe_ids_valid",
        "first_action_correct",
    ]
    out = {"baseline": {}, "finetuned": {}}
    for side, key_name in [("baseline", "baseline_score"), ("finetuned", "finetuned_score")]:
        for m in metrics:
            vals = [item[key_name][m] for item in comp_list if item[key_name][m] is not None]
            out[side][m] = {
                "passed": sum(bool(v) for v in vals),
                "total": len(vals),
                "rate": round(sum(bool(v) for v in vals) / len(vals), 4) if vals else None,
            }
    return out


comparison_all = {
    "legacy_8": build_split_comparison(baseline_legacy_results, finetuned_legacy_results, legacy_test_rows),
    "new_100": build_split_comparison(baseline_new_results, finetuned_new_results, new_test_rows),
}

summary_all = {
    "legacy_8": summarize_scores(comparison_all["legacy_8"]),
    "new_100": summarize_scores(comparison_all["new_100"]),
}

Path("/content/pilot_comparison.json").write_text(
    json.dumps(comparison_all, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
Path("/content/evaluation_summary.json").write_text(
    json.dumps(summary_all, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("=== 평가 요약 리포트 (Legacy 8 vs New 100 분리 집계) ===")
print(json.dumps(summary_all, ensure_ascii=False, indent=2))


## 11. 최종 결과 ZIP 패키징 및 다운로드


In [ ]:
import shutil
from google.colab import files

result_root = Path("/content/qwen35_864_result_v2")
result_root.mkdir(exist_ok=True)

for file_name in [
    "baseline_results.json",
    "finetuned_results.json",
    "pilot_comparison.json",
    "training_metrics.json",
    "evaluation_summary.json",
]:
    shutil.copy2(Path("/content") / file_name, result_root / file_name)

shutil.copy2(data_root / "dataset_manifest.json", result_root / "dataset_manifest.json")

if CACHE_DIR.exists():
    shutil.copytree(CACHE_DIR, result_root / "eval_cache_v2", dirs_exist_ok=True)

shutil.copytree(
    ADAPTER_DIR,
    result_root / "qwen35-fashion-864-lora-v2",
    dirs_exist_ok=True,
)

archive = shutil.make_archive(
    "/content/qwen35_fashion_864_result_v2",
    "zip",
    result_root,
)

print("결과 압축 완료:", archive)
files.download(archive)
